# Bronze — Customers
**GlobalMart Orchestration Lab**

| | |
|---|---|
| **Source** | `{mount_point}/{source_folder}/` — same mount, `orchestration-lab/customers/` subfolder |
| **Target** | `{catalog}.bronze.customers` |
| **Pattern** | Autoloader (`cloudFiles`) — same mechanism as `Databricks/Day4/Day4_3_HOL1_Build_Bronze_Layer_Mounting.ipynb`, not `COPY INTO` |

This notebook is designed to run as a **Databricks Workflow task**. Widget values are overridden at runtime by Job parameters; running it manually uses the defaults below.

**Assumes the mount from Day 4 already exists** (`dbutils.fs.mount(...)` against your real storage account) — this notebook only *verifies* it, it never re-mounts or re-embeds a storage key. If you haven't mounted yet, go run Day 4's mounting cell first.

## Step 1 — Setup: Widgets, Mount Verification, Paths

In [ ]:
from pyspark.sql.functions import col, current_timestamp

dbutils.widgets.text('catalog',       'your_catalog')
dbutils.widgets.text('schema',        'bronze')
dbutils.widgets.text('mount_point',   '/mnt/YOUR_NAME_gbmart_data')
dbutils.widgets.text('source_folder', 'orchestration-lab/customers')
dbutils.widgets.text('batch_id',      'batch_001')

CATALOG       = dbutils.widgets.get('catalog')
SCHEMA        = dbutils.widgets.get('schema')
MOUNT_POINT   = dbutils.widgets.get('mount_point')
SOURCE_FOLDER = dbutils.widgets.get('source_folder')
BATCH_ID      = dbutils.widgets.get('batch_id')

TABLE        = 'customers'
TARGET_TABLE = f'{CATALOG}.{SCHEMA}.{TABLE}'

SOURCE_PATH     = f'{MOUNT_POINT}/{SOURCE_FOLDER}/'
CHECKPOINT_PATH = f'{MOUNT_POINT}/_checkpoints/{SOURCE_FOLDER}/'
SCHEMA_PATH     = f'{MOUNT_POINT}/_schemas/{SOURCE_FOLDER}/'

print(f'Source      : {SOURCE_PATH}')
print(f'Target table: {TARGET_TABLE}')
print(f'Checkpoint  : {CHECKPOINT_PATH}')
print(f'Schema      : {SCHEMA_PATH}')

In [ ]:
# Verify the mount exists -- never re-mount here (mounts are workspace-level,
# not per-cluster, so Day 4's mount is already visible to this job cluster).
if not any(m.mountPoint == MOUNT_POINT for m in dbutils.fs.mounts()):
    raise RuntimeError(
        f"{MOUNT_POINT} is not mounted. Run Day 4's mounting notebook first -- "
        "this notebook deliberately never re-embeds a storage account key."
    )
print(f"OK -- {MOUNT_POINT} is mounted.")

In [ ]:
files = dbutils.fs.ls(SOURCE_PATH)
print(f"Files found in {SOURCE_FOLDER}/:\n")
for f in files:
    print(f"  {f.name}  ({f.size / 1024:.1f} KB)")

In [ ]:
spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
print(f"Catalog '{CATALOG}' and schema '{SCHEMA}' are ready.")

## Step 2 — Ingest with Autoloader

Same `cloudFiles` mechanism as every other Bronze table in this course. `trigger(availableNow=True)` processes everything currently sitting in the source folder, then stops — safe to re-run after dropping new files, since Autoloader's own checkpoint tracks what it already ingested.

Columns arriving: `customer_id, first_name, last_name, email, city, state, created_at` (already snake_case in the source CSV -- no PascalCase-to-snake_case rename needed in Silver, unlike the main course's historical Bronze tables).

In [ ]:
from pyspark.sql.functions import lit

customers_stream_df = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format",              "csv")
        .option("cloudFiles.schemaLocation",      SCHEMA_PATH)
        .option("cloudFiles.inferColumnTypes",    "true")
        .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
        .option("header",                         "true")
        .load(SOURCE_PATH)
        .withColumn("_source_file", col("_metadata.file_path"))
        .withColumn("_batch_id",    lit(BATCH_ID))
        .withColumn("_ingested_at", current_timestamp())
)

In [ ]:
(customers_stream_df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", CHECKPOINT_PATH)
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable(TARGET_TABLE)
)

print("Autoloader stream complete (availableNow -- processed everything new, then stopped).")

## Step 3 — Verify

In [ ]:
bronze_df = spark.table(TARGET_TABLE)
print(f"Total rows in {TARGET_TABLE}: {bronze_df.count():,}")
bronze_df.orderBy(col("_ingested_at").desc()).display()

In [ ]:
# Row count per source file -- confirms which batch(es) actually landed
bronze_df.groupBy("_source_file").count().orderBy("_source_file").display()